# Conversion Rate vs. Contact Metrics by CDP

This notebook analyzes the relationship between conversion rates and various contact metrics, grouped by CDP (Customer Direct Program).

**Charts included:**
1. Conversion Rate vs. % Under 30 Minutes Contact
2. Conversion Rate vs. % Counter Contact
3. Conversion Rate vs. % No Contact
4. Conversion Rate vs. % Under 1 Hour Contact

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
# Load data
file_path = '../data/raw/Conversion Data Nov-Dec 2025 (1).xlsx'
df = pd.read_excel(file_path, engine='openpyxl')

# Clean column names
df.columns = df.columns.str.strip()

print(f"Total records: {len(df):,}")
df.head()

In [ ]:
# Check contact time and group distributions
print("CONTACT RANGE Distribution:")
print(df['CONTACT RANGE'].value_counts())
print("\nCONTACT_GROUP Distribution:")
print(df['CONTACT_GROUP'].value_counts())

In [ ]:
# Calculate metrics by CDP
cdp_stats = df.groupby('CDP NAME').agg(
    total_reservations=('RES_ID', 'sum'),
    conversions=('RENT_IND', 'sum'),
    under_30min_count=('CONTACT RANGE', lambda x: (x == '(a)<30min').sum()),
    under_1hr_count=('CONTACT RANGE', lambda x: (x.isin(['(a)<30min', '(b)31min - 1hr'])).sum()),
    no_contact_count=('CONTACT RANGE', lambda x: (x == 'NO CONTACT').sum()),
    counter_count=('CONTACT_GROUP', lambda x: (x == 'COUNTER').sum()),
    total_contacts=('CONTACT RANGE', 'count')
).reset_index()

# Calculate rates
cdp_stats['conversion_rate'] = (cdp_stats['conversions'] / cdp_stats['total_reservations'] * 100)
cdp_stats['pct_under_30min'] = (cdp_stats['under_30min_count'] / cdp_stats['total_contacts'] * 100)
cdp_stats['pct_under_1hr'] = (cdp_stats['under_1hr_count'] / cdp_stats['total_contacts'] * 100)
cdp_stats['pct_no_contact'] = (cdp_stats['no_contact_count'] / cdp_stats['total_contacts'] * 100)
cdp_stats['pct_counter'] = (cdp_stats['counter_count'] / cdp_stats['total_contacts'] * 100)

# Filter to CDPs with meaningful volume (at least 50 reservations)
cdp_stats_filtered = cdp_stats[cdp_stats['total_reservations'] >= 50].copy()

print(f"Total CDPs: {len(cdp_stats)}")
print(f"CDPs with 50+ reservations: {len(cdp_stats_filtered)}")

cdp_stats_filtered.sort_values('total_reservations', ascending=False).head(15)

In [ ]:
# Calculate correlations for all metrics
correlations = {
    '% Under 30min': cdp_stats_filtered['pct_under_30min'].corr(cdp_stats_filtered['conversion_rate']),
    '% Under 1hr': cdp_stats_filtered['pct_under_1hr'].corr(cdp_stats_filtered['conversion_rate']),
    '% Counter': cdp_stats_filtered['pct_counter'].corr(cdp_stats_filtered['conversion_rate']),
    '% No Contact': cdp_stats_filtered['pct_no_contact'].corr(cdp_stats_filtered['conversion_rate'])
}

print("Correlations with Conversion Rate (CDP Level):")
for metric, corr in correlations.items():
    print(f"  {metric}: {corr:+.3f}")

## Chart 1: Conversion Rate vs. % Under 30 Minutes Contact

In [ ]:
# Chart 1: Conversion Rate vs. % Under 30 Minutes
fig, ax = plt.subplots(figsize=(12, 8))

sizes = cdp_stats_filtered['total_reservations'] / cdp_stats_filtered['total_reservations'].max() * 200 + 20

scatter = ax.scatter(
    cdp_stats_filtered['pct_under_30min'],
    cdp_stats_filtered['conversion_rate'],
    s=sizes,
    alpha=0.6,
    c=cdp_stats_filtered['conversion_rate'],
    cmap='RdYlGn',
    edgecolors='black',
    linewidth=0.5
)

cbar = plt.colorbar(scatter, ax=ax)
cbar.set_label('Conversion Rate (%)', fontsize=11)

ax.set_xlabel('% of Contacts Under 30 Minutes', fontsize=12)
ax.set_ylabel('Conversion Rate (%)', fontsize=12)
ax.set_title('Conversion Rate vs. Quick Contact Rate by CDP\n(bubble size = reservation volume)', fontsize=14)

ax.grid(True, alpha=0.3)

avg_conversion = cdp_stats_filtered['conversion_rate'].mean()
avg_under30 = cdp_stats_filtered['pct_under_30min'].mean()
ax.axhline(y=avg_conversion, color='red', linestyle='--', alpha=0.5, label=f'Avg Conversion: {avg_conversion:.1f}%')
ax.axvline(x=avg_under30, color='blue', linestyle='--', alpha=0.5, label=f'Avg <30min: {avg_under30:.1f}%')
ax.legend(loc='lower right')

corr = cdp_stats_filtered['pct_under_30min'].corr(cdp_stats_filtered['conversion_rate'])
ax.text(0.02, 0.98, f'Correlation: {corr:.3f}', transform=ax.transAxes, fontsize=10, verticalalignment='top')

plt.tight_layout()
plt.show()

## Chart 2: Conversion Rate vs. % Counter Contact

In [ ]:
# Chart 2: Conversion Rate vs. % Counter
fig, ax = plt.subplots(figsize=(12, 8))

sizes = cdp_stats_filtered['total_reservations'] / cdp_stats_filtered['total_reservations'].max() * 200 + 20

scatter = ax.scatter(
    cdp_stats_filtered['pct_counter'],
    cdp_stats_filtered['conversion_rate'],
    s=sizes,
    alpha=0.6,
    c=cdp_stats_filtered['conversion_rate'],
    cmap='RdYlGn',
    edgecolors='black',
    linewidth=0.5
)

cbar = plt.colorbar(scatter, ax=ax)
cbar.set_label('Conversion Rate (%)', fontsize=11)

ax.set_xlabel('% of Contacts via Counter', fontsize=12)
ax.set_ylabel('Conversion Rate (%)', fontsize=12)
ax.set_title('Conversion Rate vs. Counter Contact Rate by CDP\n(bubble size = reservation volume)', fontsize=14)

ax.grid(True, alpha=0.3)

avg_counter = cdp_stats_filtered['pct_counter'].mean()
ax.axhline(y=avg_conversion, color='red', linestyle='--', alpha=0.5, label=f'Avg Conversion: {avg_conversion:.1f}%')
ax.axvline(x=avg_counter, color='blue', linestyle='--', alpha=0.5, label=f'Avg Counter: {avg_counter:.1f}%')
ax.legend(loc='lower right')

corr = cdp_stats_filtered['pct_counter'].corr(cdp_stats_filtered['conversion_rate'])
ax.text(0.02, 0.98, f'Correlation: {corr:.3f}', transform=ax.transAxes, fontsize=10, verticalalignment='top')

plt.tight_layout()
plt.show()

## Chart 3: Conversion Rate vs. % No Contact

In [ ]:
# Chart 3: Conversion Rate vs. % No Contact (0-20% range)
fig, ax = plt.subplots(figsize=(12, 8))

# Filter to CDPs with <=20% no contact
no_contact_filtered = cdp_stats_filtered[cdp_stats_filtered['pct_no_contact'] <= 20].copy()

sizes = no_contact_filtered['total_reservations'] / no_contact_filtered['total_reservations'].max() * 200 + 20

scatter = ax.scatter(
    no_contact_filtered['pct_no_contact'],
    no_contact_filtered['conversion_rate'],
    s=sizes,
    alpha=0.6,
    c=no_contact_filtered['conversion_rate'],
    cmap='RdYlGn',
    edgecolors='black',
    linewidth=0.5
)

cbar = plt.colorbar(scatter, ax=ax)
cbar.set_label('Conversion Rate (%)', fontsize=11)

ax.set_xlabel('% of No Contact', fontsize=12)
ax.set_ylabel('Conversion Rate (%)', fontsize=12)
ax.set_title('Conversion Rate vs. No Contact Rate by CDP\n(bubble size = reservation volume, 0-20% no contact range)', fontsize=14)

ax.set_xlim(0, 20)

ax.grid(True, alpha=0.3)

avg_no_contact = no_contact_filtered['pct_no_contact'].mean()
avg_conv_no_contact = no_contact_filtered['conversion_rate'].mean()
ax.axhline(y=avg_conv_no_contact, color='red', linestyle='--', alpha=0.5, label=f'Avg Conversion: {avg_conv_no_contact:.1f}%')
ax.axvline(x=avg_no_contact, color='blue', linestyle='--', alpha=0.5, label=f'Avg No Contact: {avg_no_contact:.1f}%')
ax.legend(loc='lower right')

corr = no_contact_filtered['pct_no_contact'].corr(no_contact_filtered['conversion_rate'])
ax.text(0.02, 0.98, f'Correlation: {corr:.3f}', transform=ax.transAxes, fontsize=10, verticalalignment='top')

plt.tight_layout()
plt.show()

print(f"Showing {len(no_contact_filtered)} CDPs with <=20% no contact (excluded {len(cdp_stats_filtered) - len(no_contact_filtered)} CDPs)")

## Chart 4: Conversion Rate vs. % Under 1 Hour Contact

In [ ]:
# Chart 4: Conversion Rate vs. % Under 1 Hour
fig, ax = plt.subplots(figsize=(12, 8))

sizes = cdp_stats_filtered['total_reservations'] / cdp_stats_filtered['total_reservations'].max() * 200 + 20

scatter = ax.scatter(
    cdp_stats_filtered['pct_under_1hr'],
    cdp_stats_filtered['conversion_rate'],
    s=sizes,
    alpha=0.6,
    c=cdp_stats_filtered['conversion_rate'],
    cmap='RdYlGn',
    edgecolors='black',
    linewidth=0.5
)

cbar = plt.colorbar(scatter, ax=ax)
cbar.set_label('Conversion Rate (%)', fontsize=11)

ax.set_xlabel('% of Contacts Under 1 Hour', fontsize=12)
ax.set_ylabel('Conversion Rate (%)', fontsize=12)
ax.set_title('Conversion Rate vs. Under 1 Hour Contact Rate by CDP\n(bubble size = reservation volume)', fontsize=14)

ax.grid(True, alpha=0.3)

avg_under_1hr = cdp_stats_filtered['pct_under_1hr'].mean()
ax.axhline(y=avg_conversion, color='red', linestyle='--', alpha=0.5, label=f'Avg Conversion: {avg_conversion:.1f}%')
ax.axvline(x=avg_under_1hr, color='blue', linestyle='--', alpha=0.5, label=f'Avg <1hr: {avg_under_1hr:.1f}%')
ax.legend(loc='lower right')

corr = cdp_stats_filtered['pct_under_1hr'].corr(cdp_stats_filtered['conversion_rate'])
ax.text(0.02, 0.98, f'Correlation: {corr:.3f}', transform=ax.transAxes, fontsize=10, verticalalignment='top')

plt.tight_layout()
plt.show()

## Summary: Top 15 CDPs by Volume

In [ ]:
# Top 15 CDPs by volume with all metrics
print("Top 15 CDPs by Volume:")
cdp_stats_filtered.nlargest(15, 'total_reservations')[
    ['CDP NAME', 'total_reservations', 'conversion_rate', 
     'pct_under_30min', 'pct_under_1hr', 'pct_counter', 'pct_no_contact']
].round(1)